# Agilent BioTek Synergy H1

The Synergy H1 is a multimode microplate reader from Agilent BioTek. It supports:

- [Absorbance](../../../capabilities/absorbance) (230--999 nm)
- [Fluorescence](../../../capabilities/fluorescence) (excitation 250--700 nm, emission 250--700 nm)
- [Luminescence](../../../capabilities/luminescence)
- [Temperature control](../../../capabilities/temperature-control) (heating only, up to 45 °C)

It communicates over USB via an FTDI serial interface.

```bash
pip install pylabrobot[ftdi]
```

See the [Synergy H1 user manual](https://cqls.oregonstate.edu/sites/cqls.oregonstate.edu/files/synergy_h1_user_manual_sd-xb000426.pdf) for hardware setup.

## Setup

In [ ]:
from pylabrobot.agilent.biotek import SynergyH1

reader = SynergyH1(name="reader")
await reader.setup()

If you have multiple FTDI devices connected, pass a `device_id` to select the correct one:

```python
reader = SynergyH1(name="reader", device_id="12345678")
```

Open the tray to load a plate, then close it. Assign a plate resource so the reader knows the well layout.

In [ ]:
from pylabrobot.resources import Cor_96_wellplate_360ul_Fb

await reader.open()
plate = Cor_96_wellplate_360ul_Fb(name="plate")
reader.plate_holder.assign_child_resource(plate)
await reader.close()

## Absorbance

The Synergy H1 supports absorbance readings from 230 to 999 nm. For the full API, see [Absorbance](../../../capabilities/absorbance).

In [ ]:
results = await reader.absorbance.read(plate, wavelength=434)

## Fluorescence

Excitation range: 250--700 nm. Emission range: 250--700 nm. Focal height range: 4.5--10.68 mm. For the full API, see [Fluorescence](../../../capabilities/fluorescence).

In [ ]:
results = await reader.fluorescence.read(
    plate, excitation_wavelength=485, emission_wavelength=528, focal_height=7.5
)

## Luminescence

Focal height range: 4.5--10.68 mm. For the full API, see [Luminescence](../../../capabilities/luminescence).

Use {class}`~pylabrobot.agilent.biotek.biotek.BioTekBackend.LuminescenceParams` to set the integration time (default 1 s).

In [ ]:
results = await reader.luminescence.read(plate, focal_height=4.5)

With a custom integration time:

In [ ]:
from pylabrobot.agilent.biotek import BioTekBackend

results = await reader.luminescence.read(
    plate,
    focal_height=4.5,
    backend_params=BioTekBackend.LuminescenceParams(integration_time=2),
)

## Shaking

The Synergy H1 supports linear and orbital shaking via the driver. Frequency is specified in mm (1--6 mm, where lower values correspond to higher CPM).

In [ ]:
await reader.driver.shake(
    shake_type=BioTekBackend.ShakeType.LINEAR,
    frequency=4,  # linear frequency in mm, 1 <= frequency <= 6
)

In [ ]:
await reader.driver.stop_shaking()

## Temperature control

The Synergy H1 supports heating up to 45 °C but does not support active cooling. For the full API, see [Temperature Control](../../../capabilities/temperature-control).

In [ ]:
await reader.temperature.set_temperature(37.0)

In [ ]:
current = await reader.temperature.request_temperature()
print(f"{current:.1f} °C")

In [ ]:
await reader.temperature.deactivate()

## Teardown

In [ ]:
await reader.stop()